In [1]:
## 1. Instalar librerías y cargar configuración

In [2]:
%pip install pandas python-dotenv azure-storage-file-datalake -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pandas as pd
import os
from dotenv import load_dotenv
from azure.storage.filedatalake import DataLakeServiceClient
from io import BytesIO
from datetime import datetime
import hashlib

# Cargar configuración
load_dotenv()

AZURE_STORAGE_ACCOUNT = os.getenv("AZURE_STORAGE_ACCOUNT")
AZURE_STORAGE_KEY = os.getenv("AZURE_STORAGE_KEY")

In [4]:
## 2. Conectar a ADLS y leer datos de Bronze

In [ ]:
# Conectar a ADLS
service_client = DataLakeServiceClient(
    account_url=f"https://{AZURE_STORAGE_ACCOUNT}.dfs.core.windows.net",
    credential=AZURE_STORAGE_KEY
)
bronze_fs = service_client.get_file_system_client("bronze")
silver_fs = service_client.get_file_system_client("silver")

# Leer todos los archivos Parquet de Bronze
dataframes = {}
for path in bronze_fs.get_paths():
    if path.name.endswith(".parquet"):
        table_name = path.name.split("/")[1].replace(".parquet", "")
        file_client = bronze_fs.get_file_client(path.name)
        df = pd.read_parquet(BytesIO(file_client.download_file().readall()))
        dataframes[table_name] = df
        print(f"{table_name}: {len(df):,} registros")

✓ crm_miembros: 100,000 registros
✓ fact_devoluciones: 100,000 registros
✓ fact_ventas: 2,020,000 registros
✓ inv_stock_diario: 1,500,000 registros
✓ mstr_articulos: 10,000 registros
✓ mstr_proveedores: 1,600 registros
✓ mstr_tiendas: 300 registros


In [6]:
## 3. Función auxiliar para subir a ADLS

In [ ]:
def upload_to_adls(df, table_name):
    """Sube dataframe a ADLS en formato Parquet con overwrite para idempotencia"""
    temp_file = f"temp_{table_name}.parquet"
    df.to_parquet(temp_file, index=False)
    
    dir_client = silver_fs.get_directory_client(table_name.upper())
    try:
        dir_client.create_directory()
    except:
        pass
    
    file_client = dir_client.get_file_client(f"{table_name}.parquet")
    with open(temp_file, "rb") as data:
        file_client.upload_data(data, overwrite=True)  # overwrite=True asegura idempotencia
    
    os.remove(temp_file)
    print(f"{table_name.upper()} cargado en Silver (overwrite=True)")

In [8]:
## 4. Transformaciones Silver

In [9]:
# Asignar dataframes
articulos = dataframes["mstr_articulos"]
proveedores = dataframes["mstr_proveedores"]
tiendas = dataframes["mstr_tiendas"]
miembros = dataframes["crm_miembros"]
ventas = dataframes["fact_ventas"]
stock = dataframes["inv_stock_diario"]
devoluciones = dataframes["fact_devoluciones"]

# Inicializar tabla de errores del pipeline
pipeline_errors = pd.DataFrame(columns=['timestamp', 'tabla', 'error', 'registro_id'])

def add_error(errors_df, table_name, error_msg, record_id=None):
    """Agrega un error a la tabla de errores"""
    new_error = pd.DataFrame([{
        'timestamp': datetime.now(),
        'tabla': table_name,
        'error': error_msg,
        'registro_id': record_id
    }])
    return pd.concat([errors_df, new_error], ignore_index=True)

def hash_pii(value):
    """Hashea datos personales con SHA256"""
    if pd.isna(value):
        return value
    return hashlib.sha256(str(value).encode()).hexdigest()[:16]

def validate_referential_integrity(fact_df, dim_df, fact_key, dim_key, table_name):
    """Valida integridad referencial entre fact y dimension"""
    valid_keys = set(dim_df[dim_key].dropna())
    invalid_mask = ~fact_df[fact_key].isin(valid_keys) & fact_df[fact_key].notna()
    invalid_records = fact_df[invalid_mask].copy()
    invalid_records['error_reason'] = f"Foreign key violation: {fact_key} not found in dimension"
    invalid_records['table_name'] = table_name
    invalid_records['error_timestamp'] = datetime.now()
    valid_records = fact_df[~invalid_mask]
    return valid_records, invalid_records

In [ ]:
# ============================================
# INSPECCIONAR COLUMNAS DE LOS DATAFRAMES
# ============================================
# Primero revisamos qué columnas tiene cada tabla para evitar errores

print("Columnas en 'miembros' (crm_miembros):")
print(miembros.columns.tolist())
print("\nColumnas en 'articulos' (mstr_articulos):")
print(articulos.columns.tolist())
print("\nColumnas en 'tiendas' (mstr_tiendas):")
print(tiendas.columns.tolist())

# ============================================
# TRANSFORMACIÓN: dim_productos
# ============================================
# Unir productos con proveedores para tener información completa
dim_productos = articulos.merge(
    proveedores[['id_proveedor', 'nombre_proveedor', 'pais', 'calificacion']],
    left_on='mstr_proveedores_id_proveedor',
    right_on='id_proveedor',
    how='left'
)
# Eliminar productos duplicados (mismo id_articulo)
dim_productos = dim_productos.drop_duplicates(subset=['id_articulo'])
# Eliminar productos sin datos críticos
dim_productos = dim_productos.dropna(subset=['id_articulo', 'nombre_producto', 'categoria', 'precio'])
# Asegurar que el precio sea numérico
dim_productos['precio'] = pd.to_numeric(dim_productos['precio'], errors='coerce')
print(f"dim_productos: {len(dim_productos):,} registros")

# ============================================
# TRANSFORMACIÓN: dim_tiendas
# ============================================
# Estandarizar tipos de tienda a mayúsculas
tipo_map = {'hipermercado': 'HIPERMERCADO', 'supermercado': 'SUPERMERCADO', 'tienda_conveniencia': 'CONVENIENCIA'}
dim_tiendas = tiendas.copy()
dim_tiendas['tipo_tienda'] = dim_tiendas['tipo_tienda'].str.lower().map(tipo_map).fillna('OTRO')
# Eliminar tiendas duplicadas
dim_tiendas = dim_tiendas.drop_duplicates(subset=['id_tienda'])
# Eliminar tiendas sin datos críticos
dim_tiendas = dim_tiendas.dropna(subset=['id_tienda', 'tipo_tienda', 'pais'])
print(f"dim_tiendas: {len(dim_tiendas):,} registros")

# ============================================
# TRANSFORMACIÓN: dim_clientes
# ============================================
dim_clientes = miembros.copy()

# Verificar si existe la columna de fecha de registro antes de procesarla
if 'fec_registro' in dim_clientes.columns:
    dim_clientes['fec_registro'] = pd.to_datetime(dim_clientes['fec_registro'], errors='coerce')
    dim_clientes['antiguedad_dias'] = (datetime.now() - dim_clientes['fec_registro']).dt.days
else:
    # Si no existe la columna, crear antiguedad_dias con valor por defecto
    dim_clientes['antiguedad_dias'] = 0
    print("Columna 'fec_registro' no encontrada, usando antiguedad_dias = 0")

# Estandarizar género
if 'genero' in dim_clientes.columns:
    dim_clientes['genero'] = dim_clientes['genero'].str.lower().map({'masculino': 'M', 'femenino': 'F'}).fillna('NO_INFORMADO')
else:
    dim_clientes['genero'] = 'NO_INFORMADO'
    print("Columna 'genero' no encontrada, usando 'NO_INFORMADO'")

# Enmascarar datos personales (PII) con hash
if 'id_miembro' in dim_clientes.columns:
    dim_clientes['id_miembro_hash'] = dim_clientes['id_miembro'].apply(hash_pii)
else:
    print("Columna 'id_miembro' no encontrada")

# Eliminar clientes duplicados
dim_clientes = dim_clientes.drop_duplicates(subset=['id_miembro'])
# Eliminar clientes sin ID
dim_clientes = dim_clientes.dropna(subset=['id_miembro'])
print(f"dim_clientes: {len(dim_clientes):,} registros")

# ============================================
# TRANSFORMACIÓN: fact_ventas
# ============================================
fact_ventas = ventas.copy()
# Convertir fecha a formato datetime
fact_ventas['fecha_hora'] = pd.to_datetime(fact_ventas['fecha_hora'], errors='coerce')
# Calcular venta neta (precio - descuento)
fact_ventas['vr_venta_neto'] = fact_ventas['precio'] - fact_ventas.get('descuento', 0)
# Reemplazar miembros vacíos con 'ANONIMO'
fact_ventas['id_miembro'] = fact_ventas['id_miembro'].fillna('ANONIMO')
# Eliminar ventas duplicadas
fact_ventas = fact_ventas.drop_duplicates(subset=['id_venta'])
# Eliminar ventas sin datos críticos
fact_ventas = fact_ventas.dropna(subset=['id_venta', 'id_articulo', 'id_tienda'])
print(f"fact_ventas: {len(fact_ventas):,} registros")

# ============================================
# TRANSFORMACIÓN: fact_inventario
# ============================================
fact_inventario = stock.copy()
# Calcular días de cobertura (stock / consumo diario estimado)
fact_inventario['cobertura_dias'] = fact_inventario['stock_fisico'] / (fact_inventario['stock_fisico'] / 14 + 1)
# Marcar alerta si cobertura es menor a 7 días
fact_inventario['alerta_quiebre'] = fact_inventario['cobertura_dias'] < 7
# Eliminar registros duplicados (misma tienda y artículo)
fact_inventario = fact_inventario.drop_duplicates(subset=['id_tienda', 'id_articulo'])
print(f"fact_inventario: {len(fact_inventario):,} registros")

# ============================================
# TRANSFORMACIÓN: fact_devoluciones
# ============================================
# Mapear códigos de motivo a descripciones legibles
motivo_map = {'DEF': 'Producto defectuoso', 'TAM': 'Talla incorrecta', 'ARR': 'Arrepentimiento', 'VEN': 'Vencido', 'ERR': 'Error pedido'}
fact_devoluciones = devoluciones.copy()
# Si existe columna motivo, agregar descripción
if 'motivo' in fact_devoluciones.columns:
    fact_devoluciones['motivo_descripcion'] = fact_devoluciones['motivo'].map(motivo_map).fillna('Otro')
# Eliminar devoluciones duplicadas
fact_devoluciones = fact_devoluciones.drop_duplicates(subset=['id_venta'])
print(f"fact_devoluciones: {len(fact_devoluciones):,} registros")

Columnas en 'miembros' (crm_miembros):
['id_miembro', 'edad', 'rango_edad', 'genero', 'pais', 'ciudad', 'activo', 'batch_id', 'carga_timestamp', 'carga_fecha']

Columnas en 'articulos' (mstr_articulos):
['id_articulo', 'nombre_producto', 'categoria', 'sku', 'codigo_barras', 'precio', 'activo', 'mstr_proveedores_id_proveedor', 'batch_id', 'carga_timestamp', 'carga_fecha']

Columnas en 'tiendas' (mstr_tiendas):
['id_tienda', 'tipo_tienda', 'pais', 'ciudad', 'centro_distribucion', 'activa', 'batch_id', 'carga_timestamp', 'carga_fecha']
dim_productos: 5,000 registros
dim_tiendas: 150 registros
⚠ Columna 'fec_registro' no encontrada, usando antiguedad_dias = 0
dim_clientes: 50,000 registros
fact_ventas: 999,888 registros
fact_inventario: 473,986 registros
fact_devoluciones: 48,811 registros


In [11]:
## 5. Generar reporte de calidad

In [12]:
# ============================================
# GENERAR REPORTE DE CALIDAD DE DATOS
# ============================================
# Este reporte nos ayuda a entender cuántos datos
# se procesaron correctamente y cuántos tuvieron problemas

# Paso 1: Registrar el tiempo de inicio para medir cuánto tarda el proceso
inicio_del_proceso = datetime.now()

# Paso 2: Crear una tabla con el resumen de cada tabla procesada
# Para cada tabla, contamos:
# - originales: cuántos registros había al principio
# - limpios: cuántos quedaron después de la limpieza
# - rechazados: cuántos se eliminaron por errores o duplicados
quality_report = pd.DataFrame([
    # Tabla de productos
    {
        'nombre_tabla': 'dim_productos',
        'registros_originales': len(articulos),
        'registros_limpios': len(dim_productos),
        'registros_rechazados': len(articulos) - len(dim_productos)
    },
    # Tabla de tiendas
    {
        'nombre_tabla': 'dim_tiendas',
        'registros_originales': len(tiendas),
        'registros_limpios': len(dim_tiendas),
        'registros_rechazados': len(tiendas) - len(dim_tiendas)
    },
    # Tabla de clientes
    {
        'nombre_tabla': 'dim_clientes',
        'registros_originales': len(miembros),
        'registros_limpios': len(dim_clientes),
        'registros_rechazados': len(miembros) - len(dim_clientes)
    },
    # Tabla de ventas
    {
        'nombre_tabla': 'fact_ventas',
        'registros_originales': len(ventas),
        'registros_limpios': len(fact_ventas),
        'registros_rechazados': len(ventas) - len(fact_ventas)
    },
    # Tabla de inventario
    {
        'nombre_tabla': 'fact_inventario',
        'registros_originales': len(stock),
        'registros_limpios': len(fact_inventario),
        'registros_rechazados': len(stock) - len(fact_inventario)
    },
    # Tabla de devoluciones
    {
        'nombre_tabla': 'fact_devoluciones',
        'registros_originales': len(devoluciones),
        'registros_limpios': len(fact_devoluciones),
        'registros_rechazados': len(devoluciones) - len(fact_devoluciones)
    },
])

# Paso 3: Calcular el porcentaje de rechazo para cada tabla
# Fórmula: (rechazados ÷ originales) × 100
quality_report['porcentaje_rechazo'] = (
    quality_report['registros_rechazados'] / 
    quality_report['registros_originales'] * 100
).round(2)  # Redondear a 2 decimales

# ============================================
# GENERAR REPORTE DE EJECUCIÓN
# ============================================
# Este reporte nos dice cuánto tardó el proceso y totales generales

# Calcular el tiempo que tomó todo el proceso
tiempo_final = datetime.now()
duracion_en_segundos = (tiempo_final - inicio_del_proceso).total_seconds()

# Crear resumen de la ejecución
execution_report = pd.DataFrame([{
    'fecha_hora': tiempo_final.strftime('%Y-%m-%d %H:%M:%S'),
    'tiempo_total_segundos': round(duracion_en_segundos, 2),
    'cantidad_tablas_procesadas': len(quality_report),
    'total_registros_originales': quality_report['registros_originales'].sum(),
    'total_registros_limpios': quality_report['registros_limpios'].sum(),
    'total_registros_eliminados': quality_report['registros_rechazados'].sum(),
    'total_errores_encontrados': len(pipeline_errors)
}])

# ============================================
# MOSTRAR LOS RESULTADOS
# ============================================
print("=" * 60)
print("REPORTE DE CALIDAD DE DATOS")
print("=" * 60)
print(quality_report.to_string(index=False))
print("\n" + "=" * 60)
print("REPORTE DE EJECUCIÓN")
print("=" * 60)
print(execution_report.to_string(index=False))
print("=" * 60)

REPORTE DE CALIDAD DE DATOS
     nombre_tabla  registros_originales  registros_limpios  registros_rechazados  porcentaje_rechazo
    dim_productos                 10000               5000                  5000               50.00
      dim_tiendas                   300                150                   150               50.00
     dim_clientes                100000              50000                 50000               50.00
      fact_ventas               2020000             999888               1020112               50.50
  fact_inventario               1500000             473986               1026014               68.40
fact_devoluciones                100000              48811                 51189               51.19

REPORTE DE EJECUCIÓN
         fecha_hora  tiempo_total_segundos  cantidad_tablas_procesadas  total_registros_originales  total_registros_limpios  total_registros_eliminados  total_errores_encontrados
2026-07-29 19:10:20                    0.0                      

In [13]:
## 6. Cargar datos a Silver

In [14]:
# Cargar todas las tablas
upload_to_adls(dim_productos, 'dim_productos')
upload_to_adls(dim_tiendas, 'dim_tiendas')
upload_to_adls(dim_clientes, 'dim_clientes')
upload_to_adls(fact_ventas, 'fact_ventas')
upload_to_adls(fact_inventario, 'fact_inventario')
upload_to_adls(fact_devoluciones, 'fact_devoluciones')
upload_to_adls(quality_report, 'quality_report')
upload_to_adls(execution_report, 'execution_report')
upload_to_adls(pipeline_errors, 'pipeline_errors')

✓ DIM_PRODUCTOS cargado en Silver (overwrite=True)
✓ DIM_TIENDAS cargado en Silver (overwrite=True)
✓ DIM_CLIENTES cargado en Silver (overwrite=True)
✓ FACT_VENTAS cargado en Silver (overwrite=True)
✓ FACT_INVENTARIO cargado en Silver (overwrite=True)
✓ FACT_DEVOLUCIONES cargado en Silver (overwrite=True)
✓ QUALITY_REPORT cargado en Silver (overwrite=True)
✓ EXECUTION_REPORT cargado en Silver (overwrite=True)
✓ PIPELINE_ERRORS cargado en Silver (overwrite=True)


In [15]:
## 7. Resumen

In [ ]:
print("=" * 50)
print("Proceso Silver completado")
print(f"{len(quality_report)} tablas procesadas")
print("=" * 50)

✓ Proceso Silver completado
✓ 6 tablas procesadas


In [17]:
print("=" * 60)
print("Proceso Silver finalizado exitosamente.")
print(f"Total de tablas procesadas: {quality_report['nombre_tabla'].nunique()}")
print(f"Total de errores de integridad: {len(pipeline_errors):,}")
print(f"Reporte de calidad generado y cargado")
print("=" * 60)

Proceso Silver finalizado exitosamente.
Total de tablas procesadas: 6
Total de errores de integridad: 0
Reporte de calidad generado y cargado
